# Internship Project 2: Classification of Pet Faces
**Dataset:** Oxford-IIIT Pet Dataset (via Kaggle)  
**Model:** ResNet50 Transfer Learning  
**Classes:** 16 pet breeds (cats & dogs)

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras import Input, Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import RandomFlip, RandomRotation, Dense, Dropout
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split

print(f"TensorFlow version: {tf.__version__}")

## 2. Load Image Paths & Names

In [ ]:
images_fp = './images'

image_names = [os.path.basename(f) for f in glob.glob(os.path.join(images_fp, '*.jpg'))]

print(f"Total images found: {len(image_names)}")
print("Sample names:", image_names[:5])

## 3. Label Encoding
Oxford-IIIT Pet naming convention: `BreedName_index.jpg`  
We drop the trailing `_index` and join the rest to get the breed label.

In [ ]:
# Map breed name -> integer class index
BREED_TO_IDX = {
    'Abyssinian':                   0,
    'Bengal':                       1,
    'Birman':                       2,
    'Bombay':                       3,
    'British Shorthair':            4,
    'Egyptian Mau':                 5,
    'american bulldog':             6,
    'american pit bull terrier':    7,
    'basset hound':                 8,
    'beagle':                       9,
    'boxer':                       10,
    'chihuahua':                   11,
    'english cocker spaniel':      12,
    'english setter':              13,
    'german shorthaired':          14,
    'great pyrenees':              15,
}

NUM_CLASSES = len(BREED_TO_IDX)

def label_encode(label: str):
    """Return integer index for a breed label, or None if unknown."""
    return BREED_TO_IDX.get(label, None)


# Quick sanity check
sample_labels = [' '.join(n.split('_')[:-1]) for n in image_names[:10]]
print("Sample labels:", sample_labels)

## 4. Build Feature & Label Arrays

In [ ]:
IMAGE_SIZE = (224, 224)

features = []
labels   = []

for name in image_names:
    label_str     = ' '.join(name.split('_')[:-1])
    label_encoded = label_encode(label_str)

    if label_encoded is None:          # skip unknown breeds
        continue

    img = load_img(os.path.join(images_fp, name))
    img = img_to_array(img, dtype='uint8')
    img = tf.image.resize_with_pad(img, *IMAGE_SIZE).numpy().astype('uint8')

    features.append(img)
    labels.append(label_encoded)

features_array = np.array(features)
labels_array   = np.array(labels)

print(f"Features shape : {features_array.shape}")
print(f"Labels shape   : {labels_array.shape}")
print(f"Class distribution:\n{pd.Series(labels_array).value_counts().sort_index()}")

## 5. One-Hot Encode Labels & Train / Val / Test Split

In [ ]:
# One-hot encode
labels_one_hot = pd.get_dummies(labels_array).values   # shape: (N, 16)

# Normalise pixel values to [0, 1]
X = features_array / 255.0
y = labels_one_hot

# 60% train | 20% val | 20% test
X_temp,  X_test,  y_temp,  y_test  = train_test_split(X,      y,      test_size=0.20, random_state=42)
X_train, X_val,   y_train, y_val   = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

print(f"Train : {X_train.shape}")
print(f"Val   : {X_val.shape}")
print(f"Test  : {X_test.shape}")

## 6. Build the ResNet50 Transfer-Learning Model

In [ ]:
# Data augmentation pipeline
data_augmentation = Sequential([
    RandomFlip("horizontal_and_vertical"),
    RandomRotation(0.2),
], name="data_augmentation")

# Load frozen ResNet50 backbone (no top, global average pooling)
base_model = ResNet50(include_top=False, pooling='avg', weights='imagenet')
base_model.trainable = False

# Assemble full model
inputs  = Input(shape=(224, 224, 3), name="input_layer")
x       = data_augmentation(inputs)
x       = preprocess_input(x)          # ResNet50-specific normalisation
x       = base_model(x, training=False)
x       = Dropout(0.2)(x)
outputs = Dense(NUM_CLASSES, activation='softmax', name="predictions")(x)

model = Model(inputs, outputs, name="ResNet50_PetClassifier")
model.summary()

## 7. Compile & Train

In [ ]:
model.compile(
    optimizer=Adam(),
    loss=CategoricalCrossentropy(),
    metrics=['accuracy']
)

EPOCHS = 10

model_history = model.fit(
    x=X_train,
    y=y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS
)

## 8. Plot Training Curves

In [ ]:
acc      = model_history.history['accuracy']
val_acc  = model_history.history['val_accuracy']
loss     = model_history.history['loss']
val_loss = model_history.history['val_loss']
epochs_range = range(EPOCHS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, acc,     label='Training Accuracy')
axes[0].plot(epochs_range, val_acc, label='Validation Accuracy')
axes[0].legend(loc='lower right')
axes[0].set_title('Training and Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')

axes[1].plot(epochs_range, loss,     label='Training Loss')
axes[1].plot(epochs_range, val_loss, label='Validation Loss')
axes[1].legend(loc='upper right')
axes[1].set_title('Training and Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')

plt.tight_layout()
plt.show()

## 9. Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

## 10. Generate & Inspect Predictions

In [ ]:
y_pred        = model.predict(X_test)
y_pred_labels = np.argmax(y_pred,  axis=1)
y_true_labels = np.argmax(y_test,  axis=1)

# Reverse lookup: index -> breed name
IDX_TO_BREED = {v: k for k, v in BREED_TO_IDX.items()}

print("Sample predictions vs ground truth:")
for i in range(10):
    pred  = IDX_TO_BREED[y_pred_labels[i]]
    true  = IDX_TO_BREED[y_true_labels[i]]
    match = "✓" if pred == true else "✗"
    print(f"  [{match}] Predicted: {pred:<30}  Actual: {true}")

## 11. Visualise Predictions on Sample Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i in range(10):
    axes[i].imshow(X_test[i])
    pred  = IDX_TO_BREED[y_pred_labels[i]]
    true  = IDX_TO_BREED[y_true_labels[i]]
    color = 'green' if pred == true else 'red'
    axes[i].set_title(f"Pred: {pred}\nTrue: {true}", color=color, fontsize=8)
    axes[i].axis('off')

plt.suptitle('Predictions on Test Images (green = correct, red = wrong)', fontsize=12)
plt.tight_layout()
plt.show()